# Classification de Panneaux de Signalisation — Approche Self-Supervised Few-Shot
## Implémentation de l'article : *Self-supervised few-shot learning for real-time traffic sign classification*
### Nguyen et al., IJAIN 2024

**Idée principale** : Entraîner un réseau de *similarité* sur des images naturelles (sans panneaux),
puis l'utiliser pour classifier n'importe quel panneau de n'importe quel pays avec seulement quelques exemples.

---
### Formulation mathématique du problème

Soit $\mathcal{S} = \{(x_i, y_i)\}_{i=1}^{N \cdot K}$ l'ensemble support ($N$ classes, $K$ exemples chacune) et $q$ une image requête.

Le classifieur few-shot prédit :
$$\hat{y} = \arg\max_c \; f_{\theta}(q, x_c)$$

où $f_{\theta} : \mathbb{R}^d \times \mathbb{R}^d \to [0,1]$ est le réseau de similarité appris de façon **self-supervised**.


In [14]:
# ── Vérification GPU ──
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
n_gpu = torch.cuda.device_count()
print(f"GPUs disponibles : {n_gpu}")
for i in range(n_gpu):
    print(f"  GPU {i} : {torch.cuda.get_device_name(i)}")
if not torch.cuda.is_available():
    print("⚠️  Active le GPU : Runtime → Change runtime type → GPU T4 x2")


Device : cuda
GPUs disponibles : 2
  GPU 0 : Tesla T4
  GPU 1 : Tesla T4


In [15]:
!pip install scipy tqdm -q

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision, torchvision.transforms as transforms
import torchvision.datasets as datasets
import numpy as np, scipy.ndimage, matplotlib
matplotlib.use('Agg')          # backend non-interactif, stable sur Kaggle
import matplotlib.pyplot as plt
import random, os, time, zipfile, urllib.request
from collections import defaultdict
from tqdm import tqdm

torch.manual_seed(42); np.random.seed(42); random.seed(42)
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus  = torch.cuda.device_count()
USE_DDP = n_gpus > 1   # True sur T4 x2

# ── Répertoires de sortie Kaggle (persistent après déconnexion) ──
WORK_DIR = '/kaggle/working'
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"✅ Imports OK | Device : {device}")
print(f"   Checkpoints → {CKPT_DIR}")


✅ Imports OK | Device : cuda
   Checkpoints → /kaggle/working/checkpoints


## 1. Formulation mathématique — Augmentation des patches

### Équation 1 — Ajustement luminosité/contraste (article, Eq. 1)
$$P \leftarrow P \cdot \alpha_{\text{contrast}} + \beta_{\text{brightness}}$$

### Équations 2–3 — Bruit gaussien spatial (article, Eq. 2–3)

La gaussienne centrée sur le pixel central $(x_c, y_c)$ du patch :
$$G(x, y) = \exp\!\left(-\frac{(x-x_c)^2 + (y-y_c)^2}{2\sigma^2}\right)$$

La probabilité qu'un pixel reçoive du bruit :
$$\text{prob}(x, y) = 1 - G(x, y)$$

**Interprétation** : centre du patch → $G=1$ → prob $= 0$ (pas de bruit) ; bords → $G \approx 0$ → prob $\approx 1$ (bruit maximal).
Cela force le réseau à se concentrer sur la **région centrale** du panneau.

### Équation 4 — Perte Binary Cross-Entropy (article, Eq. 4)
$$\mathcal{L} = -\left[y \log \hat{f} + (1-y) \log(1-\hat{f})\right]$$

où $y=1$ pour une paire positive (même classe) et $y=0$ pour une paire négative.


In [16]:
# ============================================================
# PARTIE A — Augmentation CORRIGÉE (torchvision, 5x plus rapide)
# ============================================================
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import numpy as np

def augmenter_patch(img, avec_bruit_gaussien=True):
    """
    Pipeline fidèle à l'article — version rapide sans scipy.
    img : Tensor [3, H, W] dans [0,1]
    """
    H, W = img.shape[1], img.shape[2]

    # Eq. 1 : contraste + luminosité
    alpha = random.uniform(0.85, 1.15)
    beta  = random.uniform(-0.10, 0.10)
    img   = torch.clamp(img * alpha + beta, 0, 1)

    # Rotation
    angle = random.uniform(-15, 15)
    img   = TF.rotate(img, angle, fill=0)

    # Translation
    tx = random.uniform(-2, 2)
    ty = random.uniform(-2, 2)
    img = TF.affine(img, angle=0, translate=[int(tx), int(ty)],
                    scale=1.0, shear=0, fill=0)

    # Scale
    scale = random.uniform(0.9, 1.1)
    img   = TF.affine(img, angle=0, translate=[0, 0],
                      scale=scale, shear=0, fill=0)

    # Distorsion élastique légère via grid_sample
    ED_alpha, ED_sigma = 1.5, 1.0
    noise_h = torch.randn(1, H, W) * ED_alpha / H
    noise_w = torch.randn(1, H, W) * ED_alpha / W
    noise_h = F.avg_pool2d(noise_h.unsqueeze(0),
                            kernel_size=3, stride=1, padding=1).squeeze(0)
    noise_w = F.avg_pool2d(noise_w.unsqueeze(0),
                            kernel_size=3, stride=1, padding=1).squeeze(0)
    base_h  = torch.linspace(-1, 1, H).view(H, 1).expand(H, W)
    base_w  = torch.linspace(-1, 1, W).view(1, W).expand(H, W)
    grid    = torch.stack([
        torch.clamp(base_w + noise_w.squeeze(0), -1, 1),
        torch.clamp(base_h + noise_h.squeeze(0), -1, 1)
    ], dim=-1).unsqueeze(0)
    img = F.grid_sample(img.unsqueeze(0), grid,
                         mode='bilinear', align_corners=True,
                         padding_mode='zeros').squeeze(0)

    # Eq. 2-3 : Bruit gaussien spatial
    if avec_bruit_gaussien:
        xc, yc  = H // 2, W // 2
        sigma_g = H / 3.0
        xs, ys  = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
        G       = np.exp(-((xs-xc)**2 + (ys-yc)**2) / (2*sigma_g**2))
        G      /= G.max()
        prob    = (1 - G) * 0.25
        masque  = torch.from_numpy(np.random.random((H, W)) < prob)
        bruit   = torch.randn(3, H, W) * 0.05
        img     = img.clone()
        img[:, masque] += bruit[:, masque]

    return torch.clamp(img, 0, 1)

# ← ALIAS : GTSRBPairDatasetRapide appelle augmenter_patch_rapide
augmenter_patch_rapide = augmenter_patch

print("✅ Fonctions d'augmentation prêtes (augmenter_patch + augmenter_patch_rapide)")

✅ Fonctions d'augmentation prêtes (augmenter_patch + augmenter_patch_rapide)


In [17]:
# ============================================================
# VISUALISATION — Figure 1 & Figure 2 pour le rapport
# ============================================================
import os, random
from collections import defaultdict
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import transforms

# ── Reconstruire imgs_par_classe ──
GTSRB_KAGGLE_TRAIN = '/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Train'
imgs_par_classe = defaultdict(list)
for cls_name in sorted(os.listdir(GTSRB_KAGGLE_TRAIN)):
    cls_path = os.path.join(GTSRB_KAGGLE_TRAIN, cls_name)
    if not os.path.isdir(cls_path): continue
    for fname in os.listdir(cls_path):
        if fname.lower().endswith(('.ppm', '.png', '.jpg')):
            imgs_par_classe[int(cls_name)].append(os.path.join(cls_path, fname))
print(f"✅ {len(imgs_par_classe)} classes chargées")

# ── Image exemple ──
cls_exemple  = list(imgs_par_classe.keys())[0]
path_exemple = imgs_par_classe[cls_exemple][0]
transform    = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor()])
img_orig     = PILImage.open(path_exemple).convert('RGB').resize((32,32))
img_tensor   = transform(PILImage.open(path_exemple).convert('RGB'))

# ============================================================
# FIGURE 1 — CASN vs CASN(−)
# ============================================================
N_AUG = 5
fig, axes = plt.subplots(2, N_AUG+1, figsize=(14,5))
fig.suptitle("Augmentation — CASN vs CASN(−)", fontsize=13, fontweight='bold')

for row, (avec_bruit, label_row) in enumerate([(True,"CASN"),(False,"CASN(−)")]):
    axes[row,0].imshow(img_orig)
    axes[row,0].set_title("Original", color='green', fontweight='bold', fontsize=9)
    axes[row,0].axis('off')
    for k in range(N_AUG):
        random.seed(k*7 + row*100)
        np.random.seed(k*7 + row*100)
        torch.manual_seed(k*7 + row*100)
        aug = augmenter_patch(img_tensor.clone(), avec_bruit_gaussien=avec_bruit)
        axes[row,k+1].imshow(aug.permute(1,2,0).numpy())
        axes[row,k+1].set_title(f"{label_row} aug {k+1}", fontsize=8)
        axes[row,k+1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/augmentation.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 1 sauvegardée → augmentation.png")

# ============================================================
# FIGURE 2 — Bruit gaussien spatial (Eq. 2-3)
# ⚠️ Paramètres amplifiés UNIQUEMENT pour la visualisation
#    La fonction augmenter_patch() n'est PAS modifiée
# ============================================================
H, W    = 32, 32
xc, yc  = H//2, W//2
sigma_g = H / 3.0

xs, ys = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
G      = np.exp(-((xs-xc)**2 + (ys-yc)**2) / (2*sigma_g**2))
G     /= G.max()
prob   = (1 - G) * 0.25   # paramètres réels de augmenter_patch

# ── Panneau 3 : amplification ×4 densité, ×6 intensité
#    → rend le gradient bords→centre visible pour le rapport
#    → N'AFFECTE PAS augmenter_patch() ──
np.random.seed(42)
torch.manual_seed(42)
img_base = img_tensor.clone()
masque_visu = torch.from_numpy(np.random.random((H,W)) < prob * 4)  # plus dense
bruit_visu  = torch.randn(3, H, W) * 0.3                            # plus visible
img_visu    = img_base.clone()
img_visu[:, masque_visu] += bruit_visu[:, masque_visu]
img_visu    = torch.clamp(img_visu, 0, 1)
diff        = torch.abs(img_visu - img_base).mean(0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12,4))
fig.suptitle("Eq. 2-3 de l'article : bruit gaussien spatial",
             fontsize=12, fontweight='bold')

im0 = axes[0].imshow(G, cmap='hot', vmin=0, vmax=1)
axes[0].set_title("G(x,y) — Gaussienne\nBlanc = centre (pas de bruit)", fontsize=9)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(prob, cmap='hot', vmin=0, vmax=1)
axes[1].set_title("prob(x,y) = 1 − G(x,y)\nBlanc = bords (bruit max)", fontsize=9)
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(diff, cmap='hot')
axes[2].set_title("Différence originale − augmentée\n(bruit concentré aux bords)", fontsize=9)
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.savefig('/kaggle/working/bruit_gaussien.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 2 sauvegardée → bruit_gaussien.png")
print("⚠️  augmenter_patch() inchangée — amplification uniquement visuelle")

✅ 43 classes chargées
✅ Figure 1 sauvegardée → augmentation.png
✅ Figure 2 sauvegardée → bruit_gaussien.png
⚠️  augmenter_patch() inchangée — amplification uniquement visuelle


## 2. Dataset d'entraînement — Approche Self-Supervised

### Pourquoi pas GTSRB pour l'entraînement ?

L'article entraîne le réseau de similarité sur le dataset **KITTI optical flow** — des paires de frames vidéo consécutives avec les correspondances de pixels calculées par optical flow.
L'idée : deux patches centrés sur le même point dans deux frames consécutives sont **similaires** (paire positive), et deux patches loin l'un de l'autre sont **différents** (paire négative).

**Avantage** : le réseau apprend la similarité visuelle **générale**, sans jamais voir de panneaux → il peut généraliser à n'importe quel pays.

### Notre approximation (GTSRB comme proxy)

Puisque KITTI optical flow nécessite un accès et un prétraitement manuel non réalisable sous contraintes Kaggle, nous utilisons **GTSRB** comme dataset de paires d'entraînement :
- Paire **positive** : deux images de la **même classe** après augmentations différentes
- Paire **négative** : images de **classes différentes**

Cette approximation respecte l'esprit self-supervised : le réseau apprend $f_\theta(A, B) \to 1$ si $A \approx B$ visuellement, et $\to 0$ sinon.
Le protocole d'évaluation reste identique à l'article (support/requêtes issus du **test set**, séparés de l'entraînement).


In [18]:
# ============================================================
# DATASET OPTIMISÉ — Images chargées en RAM au départ
# ============================================================
import os
import random
from collections import defaultdict
from PIL import Image as PILImage

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

GTSRB_KAGGLE_TRAIN = '/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Train'

# Collecter toutes les images et construire imgs_par_classe
print("Collecte des chemins GTSRB...")
imgs_par_classe = defaultdict(list)
for cls_name in sorted(os.listdir(GTSRB_KAGGLE_TRAIN)):
    cls_path = os.path.join(GTSRB_KAGGLE_TRAIN, cls_name)
    if not os.path.isdir(cls_path):
        continue
    for fname in os.listdir(cls_path):
        if fname.lower().endswith(('.ppm', '.png', '.jpg')):
            imgs_par_classe[int(cls_name)].append(
                os.path.join(cls_path, fname))

print(f"✅ {sum(len(v) for v in imgs_par_classe.values())} images trouvées")
print(f"   {len(imgs_par_classe)} classes")


class GTSRBPairDatasetRapide(Dataset):
    """
    VERSION RAPIDE :
    - Images chargées en RAM au départ (évite I/O disque)
    - Augmentation cv2 (3× plus rapide que scipy)
    """
    def __init__(self, imgs_par_classe, n_pairs=400000, avec_bruit=True):
        self.avec_bruit = avec_bruit
        self.n_pairs    = n_pairs

        print("Chargement des images en RAM...")
        self.ipc = {}
        transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.ToTensor()
        ])
        total = 0
        for cls, paths in imgs_par_classe.items():
            imgs = []
            for p in paths:
                try:
                    img = PILImage.open(p).convert('RGB')
                    imgs.append(transform(img))
                    total += 1
                except:
                    pass
            self.ipc[cls] = imgs

        self.classes = list(self.ipc.keys())
        print(f"✅ {total} images chargées en RAM")
        print(f"   {len(self.classes)} classes")
        print(f"   {n_pairs} paires à générer")

    def __len__(self):
        return self.n_pairs

    def __getitem__(self, idx):
        if random.random() > 0.5:
            c = random.choice(self.classes)
            imgs = self.ipc[c]
            img1, img2 = random.choices(imgs, k=2)
            label = 1.0
        else:
            c1, c2 = random.sample(self.classes, 2)
            img1 = random.choice(self.ipc[c1])
            img2 = random.choice(self.ipc[c2])
            label = 0.0

        t1 = augmenter_patch_rapide(img1, avec_bruit_gaussien=self.avec_bruit)
        t2 = augmenter_patch_rapide(img2, avec_bruit_gaussien=self.avec_bruit)
        return t1, t2, torch.tensor(label, dtype=torch.float32)


# Créer les datasets
print("Création des datasets (400K paires)...")
pair_dataset       = GTSRBPairDatasetRapide(imgs_par_classe, n_pairs=400000, avec_bruit=True)
pair_dataset_minus = GTSRBPairDatasetRapide(imgs_par_classe, n_pairs=400000, avec_bruit=False)

pair_loader = DataLoader(
    pair_dataset, batch_size=128, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True,
    prefetch_factor=2)
pair_loader_minus = DataLoader(
    pair_dataset_minus, batch_size=128, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True,
    prefetch_factor=2)

print(f"✅ Datasets prêts")
print(f"   Batches par epoch : {len(pair_loader)}")

Collecte des chemins GTSRB...
✅ 39209 images trouvées
   43 classes
Création des datasets (400K paires)...
Chargement des images en RAM...
✅ 39208 images chargées en RAM
   43 classes
   400000 paires à générer
Chargement des images en RAM...
✅ 39209 images chargées en RAM
   43 classes
   400000 paires à générer
✅ Datasets prêts
   Batches par epoch : 3125


## 3. Architecture CASN — Center-Awareness Similarity Network

### Principe (Section 3.2 de l'article)

L'image $32 \times 32$ est découpée en **5 patches** de $16 \times 16$ :

$$P_1 = [0{:}16, 0{:}16], \quad P_2 = [0{:}16, 16{:}32], \quad P_3 = [16{:}32, 0{:}16], \quad P_4 = [16{:}32, 16{:}32]$$
$$P_5 = [8{:}24, 8{:}24] \quad \text{(patch central, chevauche les 4 autres)}$$

Chaque patch est analysé par le **même CNN** (poids partagés) → vecteur $\mathbf{v}_i \in \mathbb{R}^{128}$.

Concaténation : $\mathbf{z}_A = [\mathbf{v}_1^A, \ldots, \mathbf{v}_5^A] \in \mathbb{R}^{640}$

Le score de similarité final :
$$f_\theta(A, B) = \sigma\!\left(\text{FC}\!\left([\mathbf{z}_A ; \mathbf{z}_B]\right)\right) \in [0,1]$$

avec $[\mathbf{z}_A ; \mathbf{z}_B] \in \mathbb{R}^{1280}$ et les couches FC : $1280 \to 512 \to 256 \to 128 \to 1$.


In [19]:
# ============================================================
# PARTIE C — Architecture CASN (fidèle à l'article, Section 3.2)
# ============================================================

class PatchCNN(nn.Module):
    """
    Sous-réseau CNN pour un patch 16×16.
    Architecture paper : 4 couches Conv (3×3, 128 filtres) + ReLU + MaxPool.
    Sortie : vecteur de 128 dimensions.
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Couche 1 : 16×16 → 8×8
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Couche 2 : 8×8 → 4×4
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Couche 3 : 4×4 → 2×2
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Couche 4 : 2×2 → features profondes
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # → [B, 128, 1, 1]

    def forward(self, x):
        # x : [B, 3, 16, 16]
        return self.pool(self.features(x)).view(x.size(0), -1)  # [B, 128]


class CASN(nn.Module):
    """
    Center-Awareness Similarity Network (article, Section 3.2).

    Pipeline pour deux images A et B :
    1. Découpage en 5 patches 16×16
    2. PatchCNN partagé → 5 × 128 = 640 features par image
    3. Concaténation [z_A ; z_B] ∈ R^1280
    4. FC(1280→512→256→128→1) + Sigmoid → score ∈ [0,1]
    """

    def __init__(self):
        super().__init__()
        self.cnn = PatchCNN()  # CNN partagé (weight sharing)
        self.fc  = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(512, 256),  nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, 128),  nn.ReLU(inplace=True),
            nn.Linear(128, 1),    nn.Sigmoid(),
        )

    def _decoupe_5_patches(self, img):
        """
        Découpe [B, 3, 32, 32] en 5 patches [B, 3, 16, 16].

        P1=haut-gauche, P2=haut-droit, P3=bas-gauche, P4=bas-droit,
        P5=central [8:24, 8:24] (chevauche les 4 coins).
        """
        p1 = img[:, :,  0:16,  0:16]  # haut-gauche
        p2 = img[:, :,  0:16, 16:32]  # haut-droit
        p3 = img[:, :, 16:32,  0:16]  # bas-gauche
        p4 = img[:, :, 16:32, 16:32]  # bas-droit
        p5 = img[:, :,  8:24,  8:24]  # centre ← clé du paper
        return [p1, p2, p3, p4, p5]

    def extraire_features(self, img):
        """Extrait le vecteur z ∈ R^640 pour une image."""
        patches = self._decoupe_5_patches(img)
        # Passe chaque patch dans le CNN partagé
        vecs = [self.cnn(p) for p in patches]   # 5 × [B, 128]
        return torch.cat(vecs, dim=1)            # [B, 640]

    def forward(self, imgA, imgB):
        """
        Calcule f_θ(A, B) ∈ [0,1].
        Score 1 = très similaires, 0 = très différents.
        """
        zA = self.extraire_features(imgA)   # [B, 640]
        zB = self.extraire_features(imgB)   # [B, 640]
        return self.fc(torch.cat([zA, zB], dim=1)).squeeze(1)  # [B]


# ── Vérification des dimensions ──
model_test = CASN()
imgA = torch.randn(4, 3, 32, 32)
imgB = torch.randn(4, 3, 32, 32)
out  = model_test(imgA, imgB)
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"✅ CASN OK | Sortie : {out.shape} | Paramètres : {n_params:,}")
print(f"   z_A shape : {model_test.extraire_features(imgA).shape}")  # [4, 640]
print(f"   Concat [z_A;z_B] : {torch.cat([model_test.extraire_features(imgA), model_test.extraire_features(imgB)], 1).shape}")


✅ CASN OK | Sortie : torch.Size([4]) | Paramètres : 1,269,121
   z_A shape : torch.Size([4, 640])
   Concat [z_A;z_B] : torch.Size([4, 1280])


## 4. Entraînement — Minimisation de la BCE Loss (Eq. 4)

$$\mathcal{L}(\theta) = -\frac{1}{N}\sum_{i=1}^{N} \left[ y_i \log f_\theta(A_i, B_i) + (1-y_i)\log(1 - f_\theta(A_i, B_i)) \right]$$

**Optimiseur** : SGD avec momentum 0.9, lr=0.004, décroissance ×0.1 à l'époque 15 (comme l'article, Section 4).


## Rechargement des modèles entraînés et visualisation des courbes d'apprentissage

Après avoir entraîné les modèles CASN et CASN($-$) sur 400 000 paires (20 époques), nous avons sauvegardé les checkpoints pour éviter de ré-entraîner à chaque exécution du notebook. Cette section charge les poids pré-entraînés depuis Kaggle Datasets et reconstruit les courbes d'apprentissage.

### Modèles rechargés

- **CASN** : modèle avec bruit gaussien spatial activé
- **CASN($-$)** : modèle sans bruit gaussien spatial (version ablative)

Les poids sont chargés depuis les fichiers `.pth` sauvegardés, puis les modèles sont basculés en mode évaluation (`eval()`) pour désactiver les couches Dropout et BatchNorm lors de l'inférence.

### Reconstruction des courbes d'apprentissage

Les checkpoints contiennent l'historique complet des pertes (BCE Loss) et des précisions d'entraînement. Ces données sont extraites et visualisées pour comparer la convergence des deux modèles.

- **BCE Loss** : évolution de la fonction de coût Binary Cross-Entropy (Éq. 4) sur les 20 époques
- **Accuracy** : évolution de la précision de classification des paires (similaires vs dissimilaires)

La ligne verticale rouge à l'époque 15 marque la réduction du taux d'apprentissage (`lr × 0.1`), comme spécifié dans l'article original.

### Résultats finaux

| Modèle | Accuracy finale |
|--------|-----------------|
| CASN   | 99.1%           |
| CASN($-$) | 99.2%         |

Les deux modèles convergent vers des performances très élevées sur la tâche de discrimination des paires. La différence de 0.1% est négligeable, ce qui montre que l'ajout du bruit gaussien spatial n'affecte pas négativement l'apprentissage sur GTSRB (intra-domaine). Son impact sera évalué sur la généralisation cross-domain (BTSC) dans la section suivante.

In [20]:
# RECHARGEMENT — pas besoin de ré-entraîner
import shutil

model_casn = CASN()
model_casn.load_state_dict(
    torch.load('/kaggle/input/datasets/fatima12ezzahra/entrain-pth/casn.pth', map_location=device))
model_casn = model_casn.to(device)
model_casn.eval()

model_casn_minus = CASN()
model_casn_minus.load_state_dict(
    torch.load('/kaggle/input/datasets/fatima12ezzahra/entrain-pth/casn_minus.pth', map_location=device))
model_casn_minus = model_casn_minus.to(device)
model_casn_minus.eval()

print("✅ Modèles rechargés !")

# Reconstruire l'historique d'entraînement depuis les checkpoints
ckpt = torch.load('/kaggle/input/datasets/fatima12ezzahra/check-pt/casn_latest.pt', map_location=device)
hist_casn = ckpt['history']

ckpt2 = torch.load('/kaggle/input/datasets/fatima12ezzahra/check-pt/casn_minus_latest.pt', map_location=device)
hist_minus = ckpt2['history']

# Afficher les courbes d'entraînement
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_casn['loss'],  'g-o', label='CASN',    linewidth=2)
ax1.plot(hist_minus['loss'], 'b--o', label='CASN(−)', linewidth=2)
ax1.set_title('BCE Loss'); ax1.set_xlabel('Epoch')
ax1.axvline(x=14, color='red', linestyle=':', alpha=0.5)
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(hist_casn['acc'],  'g-o', label='CASN',    linewidth=2)
ax2.plot(hist_minus['acc'], 'b--o', label='CASN(−)', linewidth=2)
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'entrainement.png'), dpi=80)
plt.show()

print("✅ Modèles rechargés — pas besoin de ré-entraîner !")
print(f"   CASN      accuracy finale : {hist_casn['acc'][-1]:.1f}%")
print(f"   CASN(−)   accuracy finale : {hist_minus['acc'][-1]:.1f}%")

## Entrainement des modèles 






In [ ]:
# ============================================================
# PARTIE D — Entraînement CASN et CASN(−)
# ============================================================
# ============================================================
# PARTIE D.0 — Système de checkpoints
# ============================================================

def sauvegarder_checkpoint(model, optimizer, scheduler, epoch, history,
                            nom, best_acc, is_best=False):
    state = {
        'epoch'       : epoch,
        'model_state' : model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(),
        'optim_state' : optimizer.state_dict(),
        'sched_state' : scheduler.state_dict(),
        'history'     : history,
        'best_acc'    : best_acc,
    }
    latest_path = os.path.join(CKPT_DIR, f'{nom}_latest.pt')
    torch.save(state, latest_path)
    if is_best:
        best_path = os.path.join(CKPT_DIR, f'{nom}_best.pt')
        torch.save(state, best_path)
        print(f"   ⭐ Meilleur modèle sauvegardé ({best_acc:.2f}%) → {best_path}")

def charger_checkpoint(model, optimizer, scheduler, nom):
    latest_path = os.path.join(CKPT_DIR, f'{nom}_latest.pt')
    if not os.path.exists(latest_path):
        print(f"   Aucun checkpoint pour '{nom}' — démarrage à zéro")
        return 1, {'loss': [], 'acc': []}, 0.0
    state = torch.load(latest_path, map_location=device)
    model.load_state_dict(state['model_state'])
    optimizer.load_state_dict(state['optim_state'])
    scheduler.load_state_dict(state['sched_state'])
    print(f"   ✅ Reprise '{nom}' depuis l'époque {state['epoch']}")
    return state['epoch'] + 1, state['history'], state['best_acc']

# Supprimer anciens checkpoints
import shutil
if os.path.exists(CKPT_DIR):
    shutil.rmtree(CKPT_DIR)
    os.makedirs(CKPT_DIR)
    print("🗑️  Anciens checkpoints supprimés")

print("✅ Système de checkpoints prêt")

# ============================================================
# PARTIE D — Entraînement CASN et CASN(−)
# ============================================================

def entrainer(model, loader, nom="CASN", n_epochs=20, lr=0.004):
    """
    Entraine le modele selon les parametres de l'article (Section 4) :
    - SGD, momentum=0.9
    - lr=0.004, x0.1 a l'epoque 15
    - 20 epochs, batch=128

    Ameliorations Kaggle (logique inchangee) :
    - Checkpoint sauvegarde apres CHAQUE epoque
    - Reprise automatique si deconnexion (relancer la cellule suffit)
    - Meilleur modele sauvegarde separement
    """
    model = model.to(device)
    # ── Multi-GPU : DataParallel sur T4 x2 ──
    if torch.cuda.device_count() > 1:
        print(f"   🚀 DataParallel activé sur {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    # Eq. 4 : Binary Cross-Entropy
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr,
                          momentum=0.9, weight_decay=1e-4)
    # Decroissance lr x0.1 a l'epoque 15 (article, Section 4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer,
                                               milestones=[15], gamma=0.1)

    # ── Reprise depuis le dernier checkpoint si disponible ──
    start_epoch, history, best_acc = charger_checkpoint(
        model, optimizer, scheduler, nom)

    if start_epoch > n_epochs:
        print(f"✅ '{nom}' deja entraine ({n_epochs} epochs). Chargement des resultats.")
        return history

    print(f"\n{'='*55}")
    print(f"  Entraînement : {nom}  (epochs {start_epoch}–{n_epochs})")
    print(f"  Device       : {device}")
    print(f"{'='*55}")

    for epoch in range(start_epoch, n_epochs + 1):
        model.train()
        total_loss = correct = total = 0

        for imgA, imgB, labels in tqdm(loader, desc=f"Ep {epoch:2d}/{n_epochs}",
                                        leave=False):
            imgA, imgB, labels = imgA.to(device), imgB.to(device), labels.to(device)

            optimizer.zero_grad()
            preds = model(imgA, imgB)        # f_θ(A,B) ∈ [0,1]
            loss  = criterion(preds, labels) # Eq. 4
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct    += ((preds > 0.5).float() == labels).sum().item()
            total      += labels.size(0)

        scheduler.step()
        avg_loss = total_loss / total
        avg_acc  = correct / total * 100
        history['loss'].append(avg_loss)
        history['acc'].append(avg_acc)

        # ── Suivi du meilleur modele ──
        is_best = avg_acc > best_acc
        if is_best:
            best_acc = avg_acc

        # ── Checkpoint apres CHAQUE epoque ──
        sauvegarder_checkpoint(model, optimizer, scheduler, epoch,
                               history, nom, best_acc, is_best=is_best)

        if epoch % 2 == 0 or epoch == 1:
            print(f"  Ep {epoch:2d} | Loss={avg_loss:.4f} | Acc={avg_acc:.1f}% | lr={scheduler.get_last_lr()[0]:.5f}")

    # Sauvegarde finale dans /kaggle/working/
    final_path = os.path.join(WORK_DIR, f'{nom}.pth')
    # Unwrap DataParallel avant sauvegarde
    state_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state_to_save, final_path)
    print(f"\n✅ Modèle sauvegardé : {final_path}")
    return history


# ── Entraîner CASN (avec bruit gaussien spatial) ──
model_casn = CASN()
hist_casn  = entrainer(model_casn, pair_loader, nom="casn", n_epochs=20)

# ── Entraîner CASN(−) (sans bruit gaussien) ──
model_casn_minus = CASN()
hist_minus = entrainer(model_casn_minus, pair_loader_minus, nom="casn_minus", n_epochs=20)

# ── Courbes d'entraînement ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_casn['loss'],  'g-o', label='CASN',    linewidth=2)
ax1.plot(hist_minus['loss'], 'b--o', label='CASN(−)', linewidth=2)
ax1.set_title('Binary Cross-Entropy Loss (Eq. 4)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax1.axvline(x=14, color='red', linestyle=':', label='lr ×0.1')

ax2.plot(hist_casn['acc'],  'g-o', label='CASN',    linewidth=2)
ax2.plot(hist_minus['acc'], 'b--o', label='CASN(−)', linewidth=2)
ax2.set_title("Accuracy sur paires d'entraînement")
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Courbes d'entraînement — Self-Supervised sur GTSRB", fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'entrainement.png'), dpi=80)
plt.show()


## Entraînement des modèles CASN et CASN($-$)

L'entraînement a été réalisé sur **400 000 paires** équilibrées (50% positives, 50% négatives) pendant **20 époques**, avec les hyperparamètres suivants :

- **Optimiseur** : SGD (momentum = 0.9, weight decay = 1e-4)
- **Learning rate initial** : 0.004
- **Scheduler** : MultiStepLR (réduction ×0.1 à l'époque 15)
- **Batch size** : 128
- **Fonction de perte** : Binary Cross-Entropy (BCE)

### Utilisation du Multi-GPU

Les deux GPU Tesla T4 disponibles ont été exploités via `DataParallel`, accélérant significativement l'entraînement.

### Sauvegarde automatique (checkpoints)

À chaque époque, un checkpoint est sauvegardé (`_latest.pt`). Lorsqu'une nouvelle meilleure précision est atteinte, le modèle est également sauvegardé séparément (`_best.pt`). Cela permet :
- De reprendre l'entraînement en cas d'interruption (utile sur Kaggle)
- De conserver le meilleur modèle pour l'évaluation

### Résultats de l'entraînement

#### CASN (avec bruit gaussien spatial)

| Époque | Loss | Accuracy | Learning rate |
|--------|------|----------|---------------|
| 1 | 0.3929 | 81.3% | 0.00400 |
| 2 | 0.2079 | 92.0% | 0.00400 |
| 4 | 0.1231 | 95.4% | 0.00400 |
| 6 | 0.0926 | 96.6% | 0.00400 |
| 8 | 0.0735 | 97.3% | 0.00400 |
| 10 | 0.0599 | 97.8% | 0.00400 |
| 12 | 0.0507 | 98.1% | 0.00400 |
| 14 | 0.0425 | 98.4% | 0.00400 |
| 16 | 0.0297 | 98.9% | 0.00040 |
| 18 | 0.0256 | 99.1% | 0.00040 |
| 20 | 0.0244 | **99.1%** | 0.00040 |

#### CASN($-$) (sans bruit gaussien spatial)

| Époque | Loss | Accuracy | Learning rate |
|--------|------|----------|---------------|
| 1 | 0.3966 | 80.8% | 0.00400 |
| 2 | 0.2024 | 92.3% | 0.00400 |
| 4 | 0.1195 | 95.6% | 0.00400 |
| 6 | 0.0876 | 96.8% | 0.00400 |
| 8 | 0.0689 | 97.5% | 0.00400 |
| 10 | 0.0559 | 97.9% | 0.00400 |
| 12 | 0.0480 | 98.3% | 0.00400 |
| 14 | 0.0408 | 98.5% | 0.00400 |
| 16 | 0.0283 | 99.0% | 0.00040 |
| 18 | 0.0245 | 99.1% | 0.00040 |
| 20 | 0.0229 | **99.2%** | 0.00040 |

### Observations

1. **Convergence rapide** : Dès l'époque 2, les deux modèles dépassent 90% de précision.
2. **Stabilité** : Les pertes diminuent régulièrement sans oscillations majeures.
3. **Impact du scheduler** : La réduction du learning rate à l'époque 15 (lr = 0.0004) affine la convergence.
4. **Performances finales** :
   - CASN : 99.1% d'accuracy
   - CASN($-$) : 99.2% d'accuracy

Les deux modèles atteignent des performances quasi identiques sur la tâche de discrimination des paires. L'ajout du bruit gaussien spatial ne dégrade pas l'apprentissage sur GTSRB (intra-domaine). Son impact sera mesuré sur la généralisation cross-domain (BTSC).

### Sauvegarde finale

Les poids finaux des deux modèles ont été sauvegardés :
- `/kaggle/working/casn.pth`
- `/kaggle/working/casn_minus.pth`

## 5. Évaluation Few-Shot — Protocole de l'article

### Protocole N-way K-shot

Pour chaque requête $q$ :
1. Construire l'ensemble support $\mathcal{S} = \{x_{c,1}, \ldots, x_{c,K}\}$ pour chaque classe $c$
2. Calculer le **score moyen** (article, Section 5, protocole K-shot) :
$$s_c = \frac{1}{K}\sum_{k=1}^K f_\theta(q, x_{c,k})$$
3. Prédire : $\hat{y} = \arg\max_c s_c$

**Métriques** : Top-1, Top-2, Top-3 accuracy (comme l'article, Section 4)

> **Note implémentation** : le score agrégé est la **moyenne des scores** $f_\theta(q, x_{c,k})$ pour chaque exemple du support, et non un score calculé sur des features moyennées. Ces deux formulations ne sont pas équivalentes.


In [ ]:
# ============================================================
# PARTIE E — Chargement des datasets de test
# ============================================================

# ── GTSRB (panneaux allemands) ──
GTSRB_KAGGLE_TRAIN = '/kaggle/input/gtsrb-german-traffic-sign/Train'
GTSRB_KAGGLE_TEST  = '/kaggle/input/gtsrb-german-traffic-sign/Test'
GTSRB_LOCAL_ROOT   = './data'

transform_32 = transforms.Compose([transforms.Resize((32,32)), transforms.ToTensor()])

def charger_gtsrb_kaggle(split_path):
    from PIL import Image
    images, labels = [], []
    for cls_name in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls_name)
        if not os.path.isdir(cls_path): continue
        try: label = int(cls_name)
        except: continue
        for fname in os.listdir(cls_path):
            if fname.lower().endswith(('.ppm', '.png', '.jpg')):
                img = Image.open(os.path.join(cls_path, fname)).convert('RGB')
                images.append((transform_32(img), label))
                labels.append(label)
    ipc = defaultdict(list)
    for i, l in enumerate(labels): ipc[l].append(i)
    return images, ipc

print("Chargement GTSRB...")
if os.path.exists(GTSRB_KAGGLE_TRAIN):
    print("  Source : Kaggle dataset (/kaggle/input/)")
    train_data_raw, ipc_train = charger_gtsrb_kaggle(GTSRB_KAGGLE_TRAIN)
    test_data_raw,  ipc_test  = charger_gtsrb_kaggle(GTSRB_KAGGLE_TEST)

    class ListDataset(torch.utils.data.Dataset):
        def __init__(self, data): self.data = data
        def __len__(self): return len(self.data)
        def __getitem__(self, i): return self.data[i]

    train_data = ListDataset(train_data_raw)
    test_data  = ListDataset(test_data_raw)
    print(f"✅ GTSRB Kaggle — Train: {len(train_data)} imgs / {len(ipc_train)} classes")
    print(f"                   Test : {len(test_data)} imgs / {len(ipc_test)} classes")
else:
    print("  Source : torchvision (téléchargement réseau)")
    test_data  = datasets.GTSRB(root=GTSRB_LOCAL_ROOT, split='test',  transform=transform_32, download=True)
    train_data = datasets.GTSRB(root=GTSRB_LOCAL_ROOT, split='train', transform=transform_32, download=True)
    ipc_test  = defaultdict(list)
    ipc_train = defaultdict(list)
    for idx in range(len(test_data)):  ipc_test[test_data[idx][1]].append(idx)
    for idx in range(len(train_data)): ipc_train[train_data[idx][1]].append(idx)
    print(f"✅ GTSRB torchvision — Train: {len(train_data)} imgs / {len(ipc_train)} classes")
    print(f"                        Test : {len(test_data)} imgs / {len(ipc_test)} classes")


# ── BTSC (panneaux belges — jamais vus à l'entraînement !) ──
print("\nBTSC : vérification disponibilité...")

BTSC_OK = False
btsc_train_imgs, btsc_train_labels = [], []
btsc_test_imgs,  btsc_test_labels  = [], []
ipc_btsc_train = defaultdict(list)
ipc_btsc_test  = defaultdict(list)

BTSC_KAGGLE = '/kaggle/input/belgian-traffic-sign'

def load_btsc_folder(path):
    from PIL import Image
    images, labels = [], []
    for cls_name in sorted(os.listdir(path)):
        cls_path = os.path.join(path, cls_name)
        if not os.path.isdir(cls_path): continue
        try: label = int(cls_name)
        except: continue
        for fname in os.listdir(cls_path):
            if fname.lower().endswith(('.ppm', '.png', '.jpg')):
                img = Image.open(os.path.join(cls_path, fname)).convert('RGB')
                images.append(transform_32(img))
                labels.append(label)
    return images, labels

if os.path.exists(BTSC_KAGGLE):
    print("  Source : Kaggle dataset")
    try:
        btsc_train_imgs, btsc_train_labels = load_btsc_folder(os.path.join(BTSC_KAGGLE, 'Training'))
        btsc_test_imgs,  btsc_test_labels  = load_btsc_folder(os.path.join(BTSC_KAGGLE, 'Testing'))
        for i, l in enumerate(btsc_train_labels): ipc_btsc_train[l].append(i)
        for i, l in enumerate(btsc_test_labels):  ipc_btsc_test[l].append(i)
        BTSC_OK = True
        print(f"✅ BTSC — Train: {len(btsc_train_imgs)} imgs / {len(ipc_btsc_train)} classes")
        print(f"          Test : {len(btsc_test_imgs)} imgs / {len(ipc_btsc_test)} classes")
    except Exception as e:
        print(f"⚠️  Erreur chargement BTSC Kaggle : {e}")
else:
    os.makedirs('./data/BTSC', exist_ok=True)
    sources = [
        ("https://btsd.ethz.ch/shareddata/BelgiumTSC/BelgiumTSC_Training.zip", "btsc_train.zip", "Training"),
        ("https://btsd.ethz.ch/shareddata/BelgiumTSC/BelgiumTSC_Testing.zip",  "btsc_test.zip",  "Testing"),
    ]
    try:
        for url, fname, split in sources:
            dest = f'./data/BTSC/{fname}'
            if not os.path.exists(dest):
                print(f"  Téléchargement {fname} (ETH Zurich)...")
                urllib.request.urlretrieve(url, dest)
            with zipfile.ZipFile(dest, 'r') as z:
                z.extractall('./data/BTSC/')
        btsc_train_imgs, btsc_train_labels = load_btsc_folder('./data/BTSC/Training')
        btsc_test_imgs,  btsc_test_labels  = load_btsc_folder('./data/BTSC/Testing')
        for i, l in enumerate(btsc_train_labels): ipc_btsc_train[l].append(i)
        for i, l in enumerate(btsc_test_labels):  ipc_btsc_test[l].append(i)
        BTSC_OK = True
        print(f"✅ BTSC — Train: {len(btsc_train_imgs)} imgs / {len(ipc_btsc_train)} classes")
        print(f"          Test : {len(btsc_test_imgs)} imgs / {len(ipc_btsc_test)} classes")
    except Exception as e:
        print(f"⚠️  BTSC non disponible ({e})")
        print("   → Évaluation cross-domain ignorée, GTSRB uniquement")
        BTSC_OK = False

In [ ]:
# ============================================================
# ÉVALUATION FEW-SHOT — PROTOCOLE CUMULATIF
# ============================================================

N_TRIALS = 10

def preprocess(x):
    if isinstance(x, tuple): x = x[0]
    if isinstance(x, torch.Tensor): return x
    return transform_32(x)

def sad(imgA, imgB):
    if not isinstance(imgA, torch.Tensor): imgA = transform_32(imgA)
    if not isinstance(imgB, torch.Tensor): imgB = transform_32(imgB)
    return 1.0 / (1.0 + torch.abs(imgA.float() - imgB.float()).sum().item())

def ncc(imgA, imgB):
    if not isinstance(imgA, torch.Tensor): imgA = transform_32(imgA)
    if not isinstance(imgB, torch.Tensor): imgB = transform_32(imgB)
    a = imgA.flatten().float(); b = imgB.flatten().float()
    a -= a.mean(); b -= b.mean()
    num = (a * b).sum()
    den = torch.sqrt((a**2).sum() * (b**2).sum()) + 1e-8
    return ((num / den + 1) / 2).item()


def evaluer_casn_cumulatif(model, support_dataset, query_dataset,
                            ipc_support, ipc_query,
                            shots_list=[1,2,3,4], n_query=500):
    """
    Protocole CUMULATIF :
    - Le support K-shot contient TOUJOURS les K-1 exemples du (K-1)-shot
    - Les features du support sont calculées UNE SEULE FOIS (max_shot)
    - Pour K-shot on utilise les K PREMIERS → monotonie garantie
    """
    model_core = model.module if isinstance(model, nn.DataParallel) else model
    model_core.eval()

    max_shot = max(shots_list)
    classes  = list(set(ipc_support.keys()) & set(ipc_query.keys()))

    # Filtrer les classes qui ont assez d'exemples dans le support
    classes = [c for c in classes if len(ipc_support[c]) >= 1]

    resultats = {K: {'top1': [], 'top2': [], 'top3': []} for K in shots_list}

    for trial in range(N_TRIALS):
        random.seed(42 + trial * 137)
        np.random.seed(42 + trial * 137)
        torch.manual_seed(42 + trial * 137)

        # ── Tirer max_shot exemples UNE SEULE FOIS par classe ──
        support_pool = {}
        for c in classes:
            idx = ipc_support[c].copy()
            random.shuffle(idx)
            support_pool[c] = idx[:min(max_shot, len(idx))]

        # ── Pool de requêtes fixe pour ce trial ──
        query_pool = [(i, c) for c in classes for i in ipc_query[c]]
        random.shuffle(query_pool)
        query_pool = query_pool[:n_query]
        if not query_pool:
            continue

        # ── Pré-calculer TOUTES les features support (offline) ──
        with torch.no_grad():
            support_feats = {}
            for c, idxs in support_pool.items():
                feats = []
                for i in idxs:
                    img  = preprocess(support_dataset[i])
                    feat = model_core.extraire_features(
                               img.unsqueeze(0).to(device))  # [1, 640]
                    feats.append(feat)
                support_feats[c] = feats  # liste de tenseurs [1,640]

        # ── Évaluer pour chaque K (utilise les K PREMIERS du pool) ──
        for K in shots_list:
            top1 = top2 = top3 = 0
            with torch.no_grad():
                for q_idx, vrai in query_pool:
                    z_q = model_core.extraire_features(
                              preprocess(query_dataset[q_idx])
                              .unsqueeze(0).to(device))  # [1, 640]

                    scores = {}
                    for c, feats in support_feats.items():
                        k_r = min(K, len(feats))
                        # Score moyen sur les k_r exemples
                        scores[c] = float(np.mean([
                            model_core.fc(
                                torch.cat([z_q, zs], dim=1)
                            ).item()
                            for zs in feats[:k_r]
                        ]))

                    ranked = sorted(scores, key=scores.get, reverse=True)
                    if ranked[0]        == vrai: top1 += 1
                    if vrai in ranked[:2]:       top2 += 1
                    if vrai in ranked[:3]:       top3 += 1

            n = len(query_pool)
            resultats[K]['top1'].append(top1 / n * 100)
            resultats[K]['top2'].append(top2 / n * 100)
            resultats[K]['top3'].append(top3 / n * 100)

    # ── Moyenne sur les trials ──
    for K in shots_list:
        for m in ['top1', 'top2', 'top3']:
            vals = resultats[K][m]
            resultats[K][m] = float(np.mean(vals)) if vals else 0.0

    # ── Vérification monotonie (warning si violation) ──
    for m in ['top1', 'top2', 'top3']:
        vals = [resultats[K][m] for K in shots_list]
        for i in range(1, len(vals)):
            if vals[i] < vals[i-1] - 0.5:  # tolérance 0.5%
                print(f"  ⚠️  Monotonie {m}: {shots_list[i-1]}-shot={vals[i-1]:.1f}% "
                      f"> {shots_list[i]}-shot={vals[i]:.1f}%")

    return resultats


def evaluer_baseline_cumulatif(score_fn, dataset, ipc,
                                shots_list=[1,2,3,4], n_query=500):
    """
    Protocole cumulatif pour SAD et NCC.
    Même garantie de monotonie : support K-shot ⊂ support (K+1)-shot.
    """
    classes  = list(ipc.keys())
    max_shot = max(shots_list)

    resultats = {K: {'top1': [], 'top2': [], 'top3': []} for K in shots_list}

    for trial in range(N_TRIALS):
        random.seed(42 + trial * 137)
        np.random.seed(42 + trial * 137)

        # ── Tirer max_shot exemples UNE SEULE FOIS ──
        support_pool = {}
        for c in classes:
            idx = ipc[c].copy()
            random.shuffle(idx)
            support_pool[c] = idx[:min(max_shot, len(idx))]

        query_pool = [(i, c) for c in classes for i in ipc[c]]
        random.shuffle(query_pool)
        query_pool = query_pool[:n_query]
        if not query_pool:
            continue

        # ── Pré-charger les images support (offline) ──
        support_imgs = {}
        for c, idxs in support_pool.items():
            support_imgs[c] = [preprocess(dataset[i]) for i in idxs]

        for K in shots_list:
            top1 = top2 = top3 = 0
            for q_idx, vrai in query_pool:
                imgQ = preprocess(dataset[q_idx])
                scores = {}
                for c, imgs in support_imgs.items():
                    k_r = min(K, len(imgs))
                    # Utilise les K PREMIERS (cumulatif)
                    scores[c] = float(np.mean([
                        score_fn(imgQ, imgs[k])
                        for k in range(k_r)
                    ]))
                ranked = sorted(scores, key=scores.get, reverse=True)
                if ranked[0]        == vrai: top1 += 1
                if vrai in ranked[:2]:       top2 += 1
                if vrai in ranked[:3]:       top3 += 1

            n = len(query_pool)
            resultats[K]['top1'].append(top1 / n * 100)
            resultats[K]['top2'].append(top2 / n * 100)
            resultats[K]['top3'].append(top3 / n * 100)

    for K in shots_list:
        for m in ['top1', 'top2', 'top3']:
            vals = resultats[K][m]
            resultats[K][m] = float(np.mean(vals)) if vals else 0.0

    # ── Vérification monotonie ──
    for m in ['top1', 'top2', 'top3']:
        vals = [resultats[K][m] for K in shots_list]
        for i in range(1, len(vals)):
            if vals[i] < vals[i-1] - 0.5:
                print(f"  ⚠️  Monotonie {m}: {shots_list[i-1]}-shot={vals[i-1]:.1f}% "
                      f"> {shots_list[i]}-shot={vals[i]:.1f}%")

    return resultats


# ── Alias pour la Partie G ──
evaluer_casn_correct     = evaluer_casn_cumulatif
evaluer_baseline_correct = evaluer_baseline_cumulatif

print("✅ preprocess, sad, ncc définis")
print("✅ evaluer_casn_cumulatif + evaluer_baseline_cumulatif prêtes")
print("✅ Alias evaluer_casn_correct / evaluer_baseline_correct OK")

In [ ]:
# ============================================================
# PARTIE G — Évaluation complète
# ============================================================
res = {}

print("Préparation données...")

# ── GTSRB ──
ipc_test_gts  = defaultdict(list)
ipc_train_gts = defaultdict(list)

for idx in range(len(test_data)):
    item  = test_data[idx]
    label = item[1] if isinstance(item, tuple) else item
    ipc_test_gts[label].append(idx)

for idx in range(len(train_data)):
    item  = train_data[idx]
    label = item[1] if isinstance(item, tuple) else item
    ipc_train_gts[label].append(idx)

print(f"✅ GTSRB test  : {len(test_data)} images, {len(ipc_test_gts)} classes")

# ── BTSC ──
if BTSC_OK:
    ipc_btsc_tr = defaultdict(list)
    ipc_btsc_te = defaultdict(list)
    for i, l in enumerate(btsc_train_labels): ipc_btsc_tr[l].append(i)
    for i, l in enumerate(btsc_test_labels):  ipc_btsc_te[l].append(i)
    print(f"✅ BTSC test   : {len(btsc_test_imgs)} images, {len(ipc_btsc_te)} classes")

# ════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("ÉVALUATION — GTSRB (panneaux allemands)")
print("="*60)

print("\n→ CASN :")
r = evaluer_casn_correct(
        model_casn, train_data, test_data,
        ipc_train_gts, ipc_test_gts,
        shots_list=[1,2,3,4], n_query=500)
for K in [1,2,3,4]:
    print(f"  CASN {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
    res[f'casn_gts_{K}'] = r[K]

print("\n→ CASN(−) :")
r = evaluer_casn_correct(
        model_casn_minus, train_data, test_data,
        ipc_train_gts, ipc_test_gts,
        shots_list=[1,2,3,4], n_query=500)
for K in [1,2,3,4]:
    print(f"  CASN(-) {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
    res[f'casn_m_gts_{K}'] = r[K]

print("\n→ SAD :")
r = evaluer_baseline_correct(
        sad, test_data, ipc_test_gts,
        shots_list=[1,2,3,4], n_query=500)
for K in [1,2,3,4]:
    print(f"  SAD {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
    res[f'sad_gts_{K}'] = r[K]

print("\n→ NCC :")
r = evaluer_baseline_correct(
        ncc, test_data, ipc_test_gts,
        shots_list=[1,2,3,4], n_query=500)
for K in [1,2,3,4]:
    print(f"  NCC {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
    res[f'ncc_gts_{K}'] = r[K]

# ════════════════════════════════════════════════════════════
if BTSC_OK:
    print("\n" + "="*60)
    print("ÉVALUATION — BTSC (cross-domain)")
    print("="*60)

    communes       = set(ipc_btsc_tr.keys()) & set(ipc_btsc_te.keys())
    ipc_btsc_tr_ok = {c: ipc_btsc_tr[c] for c in communes}
    ipc_btsc_te_ok = {c: ipc_btsc_te[c] for c in communes}
    print(f"Classes communes : {len(communes)}")

    print("\n→ CASN :")
    r = evaluer_casn_correct(
            model_casn,
            btsc_train_imgs, btsc_test_imgs,
            ipc_btsc_tr_ok, ipc_btsc_te_ok,
            shots_list=[1,2,3,4], n_query=300)
    for K in [1,2,3,4]:
        print(f"  CASN {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
        res[f'casn_btsc_{K}'] = r[K]

    print("\n→ CASN(−) :")
    r = evaluer_casn_correct(
            model_casn_minus,
            btsc_train_imgs, btsc_test_imgs,
            ipc_btsc_tr_ok, ipc_btsc_te_ok,
            shots_list=[1,2,3,4], n_query=300)
    for K in [1,2,3,4]:
        print(f"  CASN(-) {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
        res[f'casn_m_btsc_{K}'] = r[K]

    print("\n→ SAD :")
    r = evaluer_baseline_correct(
            sad, btsc_test_imgs, ipc_btsc_te_ok,
            shots_list=[1,2,3,4], n_query=300)
    for K in [1,2,3,4]:
        print(f"  SAD {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
        res[f'sad_btsc_{K}'] = r[K]

    print("\n→ NCC :")
    r = evaluer_baseline_correct(
            ncc, btsc_test_imgs, ipc_btsc_te_ok,
            shots_list=[1,2,3,4], n_query=300)
    for K in [1,2,3,4]:
        print(f"  NCC {K}-shot | Top1={r[K]['top1']:.1f}%  Top2={r[K]['top2']:.1f}%  Top3={r[K]['top3']:.1f}%")
        res[f'ncc_btsc_{K}'] = r[K]

print("\n✅ Évaluation terminée !")

## Évaluation des modèles en few-shot

L'évaluation a été réalisée selon le protocole $N$-way $K$-shot avec $K \in \{1,2,3,4\}$ exemples par classe. Les métriques Top-1, Top-2 et Top-3 accuracy sont rapportées pour chaque configuration.

### GTSRB (intra-domaine) - Panneaux allemands

| Modèle | Shots | Top-1 | Top-2 | Top-3 |
|--------|-------|-------|-------|-------|
| SAD | 1 | 13.2% | 18.0% | 21.3% |
| SAD | 2 | 16.1% | 21.7% | 26.7% |
| SAD | 3 | 18.4% | 23.6% | 28.5% |
| SAD | 4 | 21.1% | 27.3% | 32.0% |
| NCC | 1 | 18.9% | 25.8% | 30.4% |
| NCC | 2 | 18.1% | 25.7% | 31.0% |
| NCC | 3 | 18.4% | 25.4% | 30.5% |
| NCC | 4 | 17.5% | 25.4% | 31.7% |
| **CASN($-$)** | 1 | **95.4%** | **97.9%** | **98.6%** |
| **CASN($-$)** | 2 | **96.0%** | **98.1%** | **98.6%** |
| **CASN($-$)** | 3 | **96.1%** | **98.2%** | **98.6%** |
| **CASN($-$)** | 4 | **96.0%** | **98.3%** | **98.7%** |
| **CASN** | 1 | **96.5%** | **98.5%** | **99.0%** |
| **CASN** | 2 | **96.8%** | **98.6%** | **99.0%** |
| **CASN** | 3 | **97.0%** | **98.7%** | **99.1%** |
| **CASN** | 4 | **97.1%** | **98.6%** | **99.2%** |

**Observations :**
- CASN surpasse largement SAD et NCC (plus de 70 points d'écart)
- Les performances sont stables quel que soit le nombre de shots
- Top-3 accuracy atteint 99.2% en 4-shot, indiquant une très haute confiance du modèle

---

### BTSC (cross-domain) - Panneaux belges (généralisation)

| Modèle | Shots | Top-1 | Top-2 | Top-3 |
|--------|-------|-------|-------|-------|
| SAD | 1 | 24.3% | 31.8% | 36.6% |
| SAD | 2 | 27.9% | 37.3% | 43.3% |
| SAD | 3 | 31.1% | 41.9% | 49.6% |
| SAD | 4 | 32.8% | 44.1% | 51.9% |
| NCC | 1 | 39.2% | 49.0% | 54.5% |
| NCC | 2 | 41.4% | 51.7% | 59.5% |
| NCC | 3 | 43.3% | 53.8% | 62.1% |
| NCC | 4 | 42.8% | 53.8% | 61.8% |
| **CASN($-$)** | 1 | **58.0%** | **70.4%** | **77.6%** |
| **CASN($-$)** | 2 | **59.4%** | **72.3%** | **79.9%** |
| **CASN($-$)** | 3 | **58.2%** | **73.2%** | **82.1%** |
| **CASN($-$)** | 4 | **59.8%** | **73.9%** | **82.5%** |
| **CASN** | 1 | **64.9%** | **73.5%** | **78.5%** |
| **CASN** | 2 | **62.7%** | **74.0%** | **80.4%** |
| **CASN** | 3 | **64.6%** | **76.2%** | **82.6%** |
| **CASN** | 4 | **64.3%** | **76.3%** | **83.9%** |

**Observations :**
- CASN surpasse nettement SAD et NCC (gain de +20 à +30 points)
- La baisse de performance par rapport à GTSRB est normale (changement de domaine)
- Le bruit gaussien spatial apporte un gain significatif (voir étude d'ablation)

✅ **Évaluation terminée**

In [ ]:
# ============================================================
# PARTIE H — Tableaux
# ============================================================

datasets_eval = [('GTSRB', 'gts')]
if BTSC_OK:
    datasets_eval.append(('BTSC (cross-domain)', 'btsc'))

for ds_name, ds_key in datasets_eval:
    print(f"\n{'='*65}")
    print(f"  TABLEAU — {ds_name}")
    print(f"{'='*65}")
    print(f"{'Méthode':10} {'Shots':>6} {'Top-1':>8} {'Top-2':>8} {'Top-3':>8}")
    print("-" * 65)
    for method, key_prefix in [('SAD','sad'),('NCC','ncc'),
                                ('CASN(−)','casn_m'),('CASN ✓','casn')]:
        for s in [1, 2, 3, 4]:
            k = f'{key_prefix}_{ds_key}_{s}'
            if k not in res: continue
            r = res[k]
            print(f"{method:10} {s:>6} {r['top1']:>7.1f}% {r['top2']:>7.1f}% {r['top3']:>7.1f}%")
        print("-" * 40)

## Tableaux récapitulatifs des performances

### Tableau GTSRB (intra-domaine)

| Méthode | Shots | Top-1 | Top-2 | Top-3 |
|---------|-------|-------|-------|-------|
| SAD | 1 | 13.2% | 18.0% | 21.3% |
| SAD | 4 | 21.1% | 27.3% | 32.0% |
| NCC | 1 | 18.9% | 25.8% | 30.4% |
| NCC | 4 | 17.5% | 25.4% | 31.7% |
| CASN($-$) | 1 | 95.4% | 97.9% | 98.6% |
| CASN($-$) | 4 | 96.0% | 98.3% | 98.7% |
| **CASN** | **1** | **96.5%** | **98.5%** | **99.0%** |
| **CASN** | **4** | **97.1%** | **98.6%** | **99.2%** |

---

### Tableau BTSC (cross-domain)

| Méthode | Shots | Top-1 | Top-2 | Top-3 |
|---------|-------|-------|-------|-------|
| SAD | 1 | 24.3% | 31.8% | 36.6% |
| SAD | 4 | 32.8% | 44.1% | 51.9% |
| NCC | 1 | 39.2% | 49.0% | 54.5% |
| NCC | 4 | 42.8% | 53.8% | 61.8% |
| CASN($-$) | 1 | 58.0% | 70.4% | 77.6% |
| CASN($-$) | 4 | 59.8% | 73.9% | 82.5% |
| **CASN** | **1** | **64.9%** | **73.5%** | **78.5%** |
| **CASN** | **4** | **64.3%** | **76.3%** | **83.9%** |

In [ ]:
# ============================================================
# PARTIE I — Graphiques finaux
# 2 figures séparées : une pour GTSRB, une pour BTSC
# ============================================================

shots = [1, 2, 3, 4]

methodes = [
    ('sad',    'r--s', 'SAD',     1.5),
    ('ncc',    'b--^', 'NCC',     1.5),
    ('casn_m', 'm-o',  'CASN(−)', 2.0),
    ('casn',   'g-o',  'CASN ✓',  2.5),
]

# ── FIGURE 1 : GTSRB ──
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 5))

for col, (metric, metric_name) in enumerate(
        [('top1','Top-1'), ('top2','Top-2'), ('top3','Top-3')]):
    ax = axes1[col]
    for key, fmt, lbl, lw in methodes:
        vals = [res.get(f'{key}_gts_{s}', {}).get(metric, 0)
                for s in shots]
        if any(v > 0 for v in vals):
            ax.plot(shots, vals, fmt, label=lbl,
                    linewidth=lw, markersize=8)
    ax.set_title(f'GTSRB — {metric_name}', fontsize=12)
    ax.set_xlabel('K-shot')
    ax.set_ylabel('Accuracy (%)')
    ax.set_xticks(shots)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle(
    'GTSRB — SAD vs NCC vs CASN(−) vs CASN\n'
    '(panneaux allemands — même domaine)',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'resultats_gtsrb.png'),
            dpi=100, bbox_inches='tight')
plt.show()
print("✅ Graphique GTSRB sauvegardé")

# ── FIGURE 2 : BTSC ──
if BTSC_OK:
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))

    for col, (metric, metric_name) in enumerate(
            [('top1','Top-1'), ('top2','Top-2'), ('top3','Top-3')]):
        ax = axes2[col]
        for key, fmt, lbl, lw in methodes:
            vals = [res.get(f'{key}_btsc_{s}', {}).get(metric, 0)
                    for s in shots]
            if any(v > 0 for v in vals):
                ax.plot(shots, vals, fmt, label=lbl,
                        linewidth=lw, markersize=8)
        ax.set_title(f'BTSC — {metric_name}', fontsize=12)
        ax.set_xlabel('K-shot')
        ax.set_ylabel('Accuracy (%)')
        ax.set_xticks(shots)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.suptitle(
        'BTSC — SAD vs NCC vs CASN(−) vs CASN\n'
        '(panneaux belges — cross-domain ⚡)',
        fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR, 'resultats_btsc.png'),
                dpi=100, bbox_inches='tight')
    plt.show()
    print("✅ Graphique BTSC sauvegardé")

print("\n✅ Tous les graphiques générés !")

In [ ]:
# ============================================================
# PARTIE J — Ablation study
# ============================================================

print("\nAblation : impact du bruit gaussien spatial (Eq. 2-3)\n")
for ds_name, ds_key in datasets_eval:
    print(f"Dataset : {ds_name}")
    print(f"{'':8} {'CASN(−)':>10} {'CASN':>10} {'Gain':>8}")
    print("-" * 40)
    for s in [1, 2]:
        for metric in ['top1', 'top2', 'top3']:
            vm = res.get(f'casn_m_{ds_key}_{s}',{}).get(metric,0)
            vc = res.get(f'casn_{ds_key}_{s}',{}).get(metric,0)
            g  = vc - vm
            print(f"{s}-shot {metric:5} {vm:>9.1f}% {vc:>9.1f}% {'↑' if g>0 else '↓'}{abs(g):>6.1f}%")
    print()

print("✅ Tout terminé")

## Étude d'ablation : impact du bruit gaussien spatial (Éq. 2-3)

L'étude d'ablation compare les performances de CASN (avec bruit gaussien spatial) et CASN($-$) (sans bruit). Le gain positif indique que le bruit améliore les performances.

### Dataset GTSRB (intra-domaine)

| Configuration | Métrique | CASN($-$) | CASN | Gain |
|---------------|----------|-----------|------|------|
| 1-shot | Top-1 | 95.4% | 96.5% | ↑ **+1.1%** |
| 1-shot | Top-2 | 97.9% | 98.5% | ↑ +0.6% |
| 1-shot | Top-3 | 98.6% | 99.0% | ↑ +0.4% |
| 2-shot | Top-1 | 96.0% | 96.8% | ↑ +0.8% |
| 2-shot | Top-2 | 98.1% | 98.6% | ↑ +0.5% |
| 2-shot | Top-3 | 98.6% | 99.0% | ↑ +0.5% |

**Sur GTSRB**, le gain est modeste (+1.1% en Top-1 1-shot), ce qui est normal car les panneaux allemands sont déjà bien centrés et peu affectés par les variations de domaine.

---

### Dataset BTSC (cross-domain)

| Configuration | Métrique | CASN($-$) | CASN | Gain |
|---------------|----------|-----------|------|------|
| 1-shot | Top-1 | 58.0% | 64.9% | ↑ **+6.9%** |
| 1-shot | Top-2 | 70.4% | 73.5% | ↑ +3.1% |
| 1-shot | Top-3 | 77.6% | 78.5% | ↑ +1.0% |
| 2-shot | Top-1 | 59.4% | 62.7% | ↑ +3.3% |
| 2-shot | Top-2 | 72.3% | 74.0% | ↑ +1.7% |
| 2-shot | Top-3 | 79.9% | 80.4% | ↑ +0.5% |

**Sur BTSC**, le gain est significatif : **+6.9 points en Top-1 1-shot**. Cela confirme que le bruit gaussien spatial améliore la généralisation cross-domain en forçant le réseau à se concentrer sur la région centrale des panneaux.

---

### Conclusion de l'ablation

| Dataset | Gain Top-1 1-shot | Interprétation |
|---------|-------------------|----------------|
| GTSRB (intra-domaine) | +1.1% | Gain modeste (domaine familier) |
| BTSC (cross-domain) | **+6.9%** | **Gain significatif** (domaine inconnu) |

✅ **Le bruit gaussien spatial améliore la généralisation cross-domain, validant l'hypothèse des auteurs.**

✅ **Tout terminé**

In [ ]:
# Note Kaggle : les fichiers dans /kaggle/working/ sont deja persistants.
# Ce bloc copie en plus vers Google Drive si vous etes sur Colab.
import shutil

# Afficher les fichiers deja sauvegardes sur Kaggle
print('Fichiers dans /kaggle/working/ :')
for fn in ['casn.pth','casn_minus.pth','augmentation.png',
           'bruit_gaussien.png','entrainement.png','resultats_finaux.png']:
    p = os.path.join(WORK_DIR, fn)
    if os.path.exists(p):
        print(f'  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')

# ── Sauvegarde Google Drive ──
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    save_dir = '/content/drive/MyDrive/Projet_Panneaux/'
    os.makedirs(save_dir, exist_ok=True)
    for f in ['casn.pth', 'casn_minus.pth', 'augmentation.png',
              'bruit_gaussien.png', 'entrainement.png', 'resultats_finaux.png']:
        if os.path.exists(f):
            shutil.copy(f, save_dir + f)
            print(f"✅ {f}")
    print(f"\n✅ Tout sauvegardé dans {save_dir}")
except:
    print("(Sauvegarde Drive ignorée — pas dans Colab)")
